In [9]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/ndai/Desktop/deidentification/deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task, _get_default_table_details_for_ui
from qc_package.scanner import DbScanner

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [10]:
qc_config = {
    "PATIENT_ID": {"prefix_value": "1001000", "length_of_value": 15},
    "ENCOUNTER_ID": {"prefix_value": "1001000", "length_of_value": 19},
}
db = DbDetailsModel.objects.get(db_name="deid_prod")

db_scanner = DbScanner(
    db.source_db_config['connection_str'],
    db.destination_db_config['connection_str'],
    db.run_config["mapping_db_config"],
    qc_config
)

In [11]:
TableDetailsModel.objects.filter(table_status=2).count()

2108

In [4]:
# code_failure = []

# for table_obj in TableDetailsModel.objects.filter(table_status=2):
#     # if table_obj.table_name in tables_for_qc:
#     if len(table_obj.qc_result) < 5:
#         print(table_obj.table_name)
#         try:
#             table_config = table_obj.table_details_for_ui
#             output_result = db_scanner.scan_table(table_obj.table_name, table_config, table_obj.id)
#             table_obj.qc_result = output_result
#             table_obj.save()
#         except Exception as e:
#             # print(traceback.format_exc())
#             code_failure.append(table_obj.table_name)

In [5]:
# len(code_failure)

In [15]:
import os, json
from datetime import datetime

all_results = []

def is_phi_table(table_config):
    for col_conf in table_config['columns_details']:
        if col_conf['is_phi']:
            return True
    return False
    
for table_obj in TableDetailsModel.objects.filter(table_status=2):
    # if table_obj.table_name in tables_for_qc:
    qc_result = table_obj.qc_result
    if not qc_result:
        continue
    one_table = {
        "autoqc_date": datetime.now().date(),
        "table_name": table_obj.table_name,
        "source_rows_count": qc_result['source_rows_count'],
        "dest_rows_count": qc_result['dest_rows_count'],
        "status": qc_result.get("final_qc_result", {}).get("is_qc_passed"),
        "ignore_rows_count": qc_result['ignore_rows_count'],
        "reason": qc_result.get("final_qc_result", {}).get("reason"),
        "is_phi": is_phi_table(table_obj.table_details_for_ui)
    }
    all_results.append(one_table)
len(all_results)

1344

In [16]:
res_df = pd.DataFrame(all_results)
res_df.shape

(1344, 8)

In [17]:
res_df['status'].value_counts()

status
True     1049
False     295
Name: count, dtype: int64

In [32]:
# batch_df = pd.read_csv('emd_b3_phi.csv')
# tables = list(batch_df['TABLE_NAME'].unique())
# len(tables)

In [33]:
# res_df = res_df[res_df['table_name'].isin(tables)]
# res_df.shape

In [ ]:
# res_df.to_csv("qc_results_12082025.csv", index=False)